In [ ]:
from sklearn.decomposition import PCA
import time
def pca(df_X, df_y):
    pca = PCA(n_components = 2)
    pca.fit(df_X)
    df_pca = pca.transform(df_X)
    df_pca = pd.DataFrame(df_pca, columns = ['comp. 0', 'comp. 1'])
    df_pca['target'] = df_y
    print('variance ratio:', pca.explained_variance_ratio_, 'sum:', sum(pca.explained_variance_ratio_))
    return df_pca

def concath(df_X, df_y):
    df = pd.concat([df_X, df_y])
    return df

In [ ]:
import tensorflow as tf
import math
pi = tf.constant(math.pi)

##############################################################################

acc = lambda TN,FP,FN,TP: tf.math.divide_no_nan(TP+TN,TN+FP+FN+TP)
prec = lambda TN,FP,FN,TP: tf.math.divide_no_nan(TP,FP+TP)
rec = lambda TN,FP,FN,TP: tf.math.divide_no_nan(TP,FN+TP)
spec = lambda TN,FP,FN,TP: tf.math.divide_no_nan(TN,FP+TN)
f1s = lambda TN,FP,FN,TP: 2.*tf.math.divide_no_nan(prec(TN,FP,FN,TP)*\
                          rec(TN,FP,FN,TP),prec(TN,FP,FN,TP)+rec(TN,FP,FN,TP))
tss = lambda TN,FP,FN,TP: rec(TN,FP,FN,TP)+spec(TN,FP,FN,TP)-1.
csi = lambda TN,FP,FN,TP: tf.math.divide_no_nan(TP,FN+FP+TP)
hss1 = lambda TN,FP,FN,TP: tf.math.divide_no_nan(TP-FP,FN+TP)
hss2 = lambda TN,FP,FN,TP: tf.math.divide_no_nan(2.*((TP*TN)-(FP*FN)),((TP+FN)\
                            *(FN+TN))+((TP+FP)*(TN+FP)))
bacc = lambda TN,FP,FN,TP: tf.math.divide_no_nan(rec(TN,FP,FN,TP)+spec(TN,FP,FN,TP),2.0)
    
F_unif = lambda x: x
F_cos = lambda x,mu,delta: tf.where(x<mu-delta,0.,tf.where(x>mu+delta,1.,\
        0.5*(1.+tf.math.divide(x-mu,delta)+1./pi*tf.math.sin(pi*\
         tf.math.divide(x-mu,delta)))))



def SOL(score = 'accuracy', distribution = 'uniform',\
        mu = 0.5, delta = 0.1, mode = 'average'):

    """
    
    Score-Oriented Loss (SOL)

    Compute the expected confusion matrix defined by the following elements
           n
      TP = ∑  y_i * F(p_i)
           n
      TN = ∑  (1 - y_i) * (1 - F(p_i))
          i=1
           n
      FP = ∑  (1 - y_i) * F(p_i)
          i=1
           n
      FN = ∑  y_i * (1 - F(p_i))
          i=1

      where y_i represents the true label, p_i represents the predicted probability given by p_i = sigmoid(x_i) and
      F represents the a priori distribution for the threshold.

      The Score-Oriented loss is defined on the elements of the expected confusion matrix as follows

      loss = - score(TP,TN,FP,FN) + 1,

      where score represents the chosen skill score.

      Example
      if score = 'accuracy'
      then
      loss = - (TP + TN) / (TP + FN + TN + FP) + 1
    
    Authors: Guastavino S. & Marchetti F.

    References: https://arxiv.org/abs/2103.15522

    Usage:
     model.compile(loss=SOL(score = 'accuracy', distribution = 'uniform', mu = 0.5, delta = 0.1, mode = 'average'))

    Parameters
    ----------
    
    score : string, the chosen score used to build the loss. Implemented 
            choices are ['accuracy','precision','recall','specificity',
            'f1_score','tss','csi','hss1','hss2'].
    
    distribution : string, the a priori distribution for the threshold.
                   Implemented choices are ['uniform','cosine'].
             
    mu : scalar in (0,1) or list of scalars in (0,1). If the chosen 
         distribution is 'cosine', mu is the mean of the raised cosine 
         distribution. In the multiclass case, mu can be defined as a list of
         values, one for each one-vs-rest classification, so that
         len(mu) = number of classes. 
         If the the chosen distribution is 'uniform', this parameter is
         ignored.
    
    delta : scalar in (0,1). If the chosen distribution is 'cosine', then
            [mu-delta,mu+delta] is the support of the raised cosine
            distribution. In the multiclass case, delta can be defined as a
            list values, one for each one-vs-rest classification, so that
            len(delta) = number of classes. 
            If the the chosen distribution is 'uniform', this parameter is
            ignored.
    
    mode : string in ['average','weighted']. In the multiclass case, it 
           determines in which way the contributes of the one-vs-rest tasks
           are combined in a unique score. 
           If the problem is not multiclass, this parameter is ignored.
    
    
    """
        
    if score == 'accuracy':
        score = acc
    if score == 'precision':
        score = prec
    if score == 'recall':
        score = rec    
    if score == 'specificity':
        score = spec        
    if score == 'f1_score':
        score = f1s
    if score == 'tss':
        score = tss
    if score == 'csi':
        score = csi
    if score == 'hss1':
        score = hss1        
    if score == 'hss2':
        score = hss2 
    if score == 'bacc':
        score = bacc
        
    if distribution == 'uniform':
        distr = F_unif
        
    if distribution == 'cosine':
        if type(mu) is not list:
            distr = lambda x: F_cos(x,mu,delta)
        else:
            distr = [lambda x: F_cos(x,mu[j],delta[j]) for j in \
                     range(0,len(mu))]
            
    def SOL_(y_true, y_pred):
                
        y_true = tf.convert_to_tensor(y_true, tf.float32)
        y_pred = tf.convert_to_tensor(y_pred, tf.float32)
        
        if y_pred.shape[1] == 1:   
            
            TN = tf.reduce_sum((1.-y_true)*(1.-distr(y_pred)))
            TP = tf.reduce_sum(y_true*distr(y_pred))
            FP = tf.reduce_sum((1.-y_true)*distr(y_pred))
            FN = tf.reduce_sum(y_true*(1.-distr(y_pred)))
        
            return -score(TN,FP,FN,TP) + 1.
        
        else:
            
            score_arr = []
            num_c0_arr = []
            
            
            for j in range(0,y_pred.shape[1]):
                                 
                y_pred_ = y_pred[:,j]
                y_true_ = y_true[:,j]
                
                if type(mu) is not list:

                    TN = tf.reduce_sum((1.-y_true_)*(1.-distr(y_pred_)))
                    TP = tf.reduce_sum(y_true_*distr(y_pred_))
                    FP = tf.reduce_sum((1.-y_true_)*distr(y_pred_))
                    FN = tf.reduce_sum(y_true_*(1.-distr(y_pred_)))
                
                else:
                
                    TN = tf.reduce_sum((1.-y_true_)*(1.-distr[j](y_pred_)))
                    TP = tf.reduce_sum(y_true_*distr[j](y_pred_))
                    FP = tf.reduce_sum((1.-y_true_)*distr[j](y_pred_))
                    FN = tf.reduce_sum(y_true_*(1.-distr[j](y_pred_)))
                    
                score_arr.append(score(TN,FP,FN,TP))

                if mode == 'weighted':
                    num_c0_arr.append(tf.cast(tf.shape(y_true_)[0],\
                                        tf.float32)-tf.reduce_sum(y_true_))
            
            score_arr = tf.stack(score_arr,axis=0)
            
            if mode == 'weighted':
                num_c0_arr = tf.stack(num_c0_arr,axis=0)
                final_score = tf.math.divide_no_nan(tf.reduce_sum\
                            (score_arr*num_c0_arr),tf.reduce_sum(num_c0_arr))
                
            if mode == 'average':
                final_score = tf.math.reduce_mean(score_arr)
            
            return -final_score + 1.
    
    return SOL_


In [ ]:
from keras.models import Sequential
from keras.layers import Dense
from tensorflow.keras.optimizers import SGD
from matplotlib import pyplot as plt
from sklearn import metrics
import tensorflow as tf
import keras
from keras.layers import BatchNormalization
from keras.layers import Activation
from keras import optimizers
import math


################################ MSE ################################
def MSE(y_true, y_pred):
    return tf.reduce_mean(tf.math.square(y_true - y_pred))

################################ BCE ################################
import tensorflow as tf
def BCE(y_true, y_pred):
    return -tf.reduce_mean(y_true*tf.math.log(y_pred)+(1-y_true)*tf.math.log(1-y_pred))

################################ Ours_Accu ################################
def Ours_Accu(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    yl = y_train.shape[0]
    accu = (yl-tf.reduce_sum(y_true)-tf.reduce_sum(y_pred)+2*tf.reduce_sum(y_true*y_pred)) / yl
    return 1-accu

################################ Ours_Fbeta ################################
def Ours_Fbeta(y_true, y_pred):
#     beta = 1 
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    numerator = (1+beta**2)*tf.reduce_sum(y_true*y_pred)
    denominator = (beta**2)*tf.reduce_sum(y_true) + tf.reduce_sum(y_pred)
    return 1-(numerator/denominator)

################################ Ours_Gmean ################################
def Ours_Gmean(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    syhy = tf.reduce_sum(y_true*y_pred)
    sy = tf.reduce_sum(y_true)
    yl = (y_train.shape[0])
#     gmean = syhy*(yl-tf.reduce_sum(y_pred)-sy+syhy)/(sy*(yl-sy))
    gmean = tf.sqrt(syhy*(yl-tf.reduce_sum(y_pred)-sy+syhy)/(sy*(yl-sy)))
    return 1-gmean

################################ Ours_BAccu ################################
def Ours_BAccu(y_true, y_pred):
    y_pred = 1/(1+tf.math.exp(-L*(y_pred-0.5)))
    syhy = tf.reduce_sum(y_true*y_pred)
    sy = tf.reduce_sum(y_true)
    yl = y_train.shape[0]
    baccu = (yl*(syhy+sy)-sy*(tf.reduce_sum(y_pred)+sy)) / (2*sy*(yl-sy))
    return 1-baccu

# 1. My own data(2d / 10,000)

In [ ]:
from sklearn import datasets
import numpy as np
import pandas as pd
Init_X, Init_y = datasets.make_classification(n_samples=10000, n_classes=2, weights=[0.9, 0.1], class_sep=1.2,
                                    n_features=5, n_informative=3, n_redundant=1, n_clusters_per_class=1, random_state=0)
X = np.array(Init_X)
y = np.array(Init_y)
# # change 0 -> -1
# y = [-1 if x==0 else x for x in y]

df_pca = pca(X, y)
df_pca

In [ ]:
T_sse= []
T_mse= []
T_bce= []
T_acc= []
T_f1= []
T_f05= []
T_f2= []
T_gmean= []
T_bacc= []
T_sola = []
T_solf = []
T_solb = []

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state = 2)

X = df_pca.iloc[:, :2]
y = df_pca.iloc[:, 2]

In [ ]:
L = 73
hidden_node = 2
# momentum=0.9
activation = 'sigmoid'  
kernel_initializer=keras.initializers.he_normal(seed=100)
epochs=100

In [ ]:
n_iter=0

###################### MLP (sigmoid // MSE) ##############################
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate = 0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_mse = time.time()
    model.compile(loss=MSE, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    
    T_mse.append(time.time()-start_mse)
print(np.mean(T_mse))    

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_bce = time.time()
    model.compile(loss=BCE, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_bce.append(time.time()-start_bce)
print(np.mean(T_bce))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Accuracy) ##############################
batch_size = int(X.shape[0]*0.9 * 0.05)  # 0.05%0.004
print('batch_size: ', batch_size)
learning_rate = 0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_acc = time.time()
    model.compile(loss=Ours_Accu, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_acc.append(time.time()-start_acc)
print(np.mean(T_acc))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // F1) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   # 0.05&0.001
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 1
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f1 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f1.append(time.time()-start_f1)
print(np.mean(T_f1))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // F0.5) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.05&0.0005
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 0.5
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f05 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f05.append(time.time()-start_f05)
print(np.mean(T_f05))    

In [ ]:
n_iter=0
    
###################### MLP (sigmoid // sigmoid // F2) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.05&0.005
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 2
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)
    
for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f2 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f2.append(time.time()-start_f2)
print(np.mean(T_f2))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Gmean) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.5&0.005
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_gmean = time.time()
    model.compile(loss=Ours_Gmean, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_gmean.append(time.time()-start_gmean)
print(np.mean(T_gmean))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Balanced Accuracy) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.5&0.005  866
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_bacc = time.time()
    model.compile(loss=Ours_BAccu, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_bacc.append(time.time()-start_bacc)
print(np.mean(T_bacc))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_sola = time.time()
    model.compile(loss=SOL(score = 'accuracy', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_sola.append(time.time()-start_sola)
print(np.mean(T_sola))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_solf = time.time()
    model.compile(loss=SOL(score = 'f1_score', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_solf.append(time.time()-start_solf)
print(np.mean(T_solf))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_pca, df_pca['target']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_solb = time.time()
    model.compile(loss=SOL(score = 'bacc', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_solb.append(time.time()-start_solb)
print(np.mean(T_solb))  

In [ ]:
times = []
names = [T_mse, T_bce, T_acc, T_f1, T_f05, T_f2, T_gmean, T_bacc, T_sola, T_solf, T_solb]
NAME = ['T_mse', 'T_bce', 'T_acc', 'T_f1', 'T_f05', 'T_f2', 'T_gmean', 'T_bacc', 'T_sola', 'T_solf', 'T_solb']
for i in range(len(names)):
    times.append(np.mean(names[i]))

rate = times/times[1]

df_time = pd.DataFrame(NAME, columns=['names'])
df_time['times'] = times
df_time['rate'] = rate
df_time

In [ ]:
lmean = np.mean([times[2],times[3],times[7]])   # ours acc, f1, bacc
print(lmean, lmean/times[1])

In [ ]:
smean = np.mean(times[8:])   # sol acc, f1, bacc
print(smean, smean/times[1])

## For Extension (2/5/25)

In [ ]:
## For Extension (2/5/25)
import numpy as np
# F1/GM/BA
# MSE/BCE/ANyLoss(Type1)
MSEtime = 3.159441
BCEtime = 3.156025
Anyf1 = 3.194723
Anygm = 3.242531
Anyba = 3.148364
print(MSEtime, BCEtime, Anyf1, Anygm, Anyba)
print(MSEtime/BCEtime, BCEtime/BCEtime, Anyf1/BCEtime, Anygm/BCEtime, Anyba/BCEtime)

# 2. Creditcard Fraud Detection 2023(29d / 298531)

In [ ]:
# class '0' = normal, class '1' = anomaly
card_df = pd.read_csv('creditcard_2023.csv')
card_df.shape

In [ ]:
card_df.isnull().sum()

In [ ]:
card_df.head()

In [ ]:
card_df.describe()

In [ ]:
# Amount values largely varies.

# # Normalization
# card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].min())/(card_df.iloc[:,:-1].max() - card_df.iloc[:,:-1].min())

# Standardization
card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].mean())/card_df.iloc[:,:-1].std()

card_df

In [ ]:
card_df['Class'].value_counts()

In [ ]:
# Data is too balanced!!! We intentionally make it imbalanced.
df_0 = card_df[card_df['Class']==0]
df_1 = card_df[card_df['Class']==1]
print(len(df_0), len(df_1))

In [ ]:
N = round(len(df_0)*0.05)
df_1_samp = df_1.sample(n=N, random_state = 100)
df_1_samp

In [ ]:
df_card = concath(df_0, df_1_samp)
df_card

In [ ]:
df_card.columns

In [ ]:
df_card = df_card.drop('id', axis=1)
df_card

In [ ]:
T_sse= []
T_mse= []
T_bce= []
T_acc= []
T_f1= []
T_f05= []
T_f2= []
T_gmean= []
T_bacc= []
T_sola = []
T_solf = []
T_solb = []

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state = 2)

X = df_card.iloc[:, :-1]
y = df_card.iloc[:, -1]

In [ ]:
L = 73
hidden_node = 2
# momentum=0.9
activation = 'sigmoid'  
kernel_initializer=keras.initializers.he_normal(seed=100)
epochs=100

In [ ]:
n_iter=0

###################### MLP (sigmoid // MSE) ##############################
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate = 0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_mse = time.time()
    model.compile(loss=MSE, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_mse.append(time.time()-start_mse)
print(np.mean(T_mse))    

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_bce = time.time()
    model.compile(loss=BCE, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_bce.append(time.time()-start_bce)
print(np.mean(T_bce))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Accuracy) ##############################
batch_size = int(X.shape[0]*0.9 * 0.05)  # 0.05%0.004
print('batch_size: ', batch_size)
learning_rate = 0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_acc = time.time()
    model.compile(loss=Ours_Accu, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_acc.append(time.time()-start_acc)
print(np.mean(T_acc))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // F1) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   # 0.05&0.001
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 1
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f1 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f1.append(time.time()-start_f1)
print(np.mean(T_f1))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // F0.5) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.05&0.0005
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 0.5
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f05 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f05.append(time.time()-start_f05)
print(np.mean(T_f05))    

In [ ]:
n_iter=0
    
###################### MLP (sigmoid // sigmoid // F2) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.05&0.005
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 2
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)
    
for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f2 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f2.append(time.time()-start_f2)
print(np.mean(T_f2))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Gmean) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.5&0.005
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_gmean = time.time()
    model.compile(loss=Ours_Gmean, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_gmean.append(time.time()-start_gmean)
print(np.mean(T_gmean))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Balanced Accuracy) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.5&0.005  866
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_bacc = time.time()
    model.compile(loss=Ours_BAccu, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_bacc.append(time.time()-start_bacc)
print(np.mean(T_bacc))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_sola = time.time()
    model.compile(loss=SOL(score = 'accuracy', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_sola.append(time.time()-start_sola)
print(np.mean(T_sola))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_solf = time.time()
    model.compile(loss=SOL(score = 'f1_score', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_solf.append(time.time()-start_solf)
print(np.mean(T_solf))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(df_card, df_card['Class']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_solb = time.time()
    model.compile(loss=SOL(score = 'f1_score', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_solb.append(time.time()-start_solb)
print(np.mean(T_solb))  

In [ ]:
times = []
names = [T_mse, T_bce, T_acc, T_f1, T_f05, T_f2, T_gmean, T_bacc, T_sola, T_solf, T_solb]
NAME = ['T_mse', 'T_bce', 'T_acc', 'T_f1', 'T_f05', 'T_f2', 'T_gmean', 'T_bacc', 'T_sola', 'T_solf', 'T_solb']
for i in range(len(names)):
    times.append(np.mean(names[i]))

rate = times/times[1]

df_time = pd.DataFrame(NAME, columns=['names'])
df_time['times'] = times
df_time['rate'] = rate
df_time

In [ ]:
lmean = np.mean([times[2],times[3],times[7]])   # ours acc, f1, bacc
print(lmean, lmean/times[1])

In [ ]:
smean = np.mean(times[8:])   # sol acc, f1, bacc
print(smean, smean/times[1])

In [ ]:
np.mean(times[2:])

In [ ]:
np.mean(times[2:]) / times[1]

In [ ]:
import numpy as np
np.mean([6.846011, 6.692287, 6.928625, 6.871951, 6.912842, 6.749057])

In [ ]:
import numpy as np
np.mean([6.846011, 6.692287, 6.928625, 6.871951, 6.912842, 6.749057]) / 6.834837

## For Extension (2/5/25)

In [ ]:
## For Extension (2/5/25)
import numpy as np
# F1/GM/BA
# MSE/BCE/ANyLoss(Type1)
MSEtime = 3.909926
BCEtime = 4.379846
Anyf1 = 4.327500
Anygm = 4.564762
Anyba = 4.319203
print(MSEtime, BCEtime, Anyf1, Anygm, Anyba)
print(MSEtime/BCEtime, BCEtime/BCEtime, Anyf1/BCEtime, Anygm/BCEtime, Anyba/BCEtime)

# 3. Breast Cancer Data (30d / 569)

In [ ]:
# class 'B' = Benign, class 'M' = Malignant
cancer_df = pd.read_csv('breast_cancer.csv')
cancer_df.shape

In [ ]:
cancer_df.isnull().sum()

In [ ]:
cancer_df.head()

In [ ]:
cancer_df.describe()

In [ ]:
# M/Malignant = 0, B/Benign = 1
y_encoded, y_class = pd.factorize(cancer_df['diagnosis'])
print(y_class)
y_encoded

In [ ]:
# But I want [B/Benign = 0(Major), M/Malignant = 1(minor)]
y_encoded = (y_encoded+1)%2
y_encoded

In [ ]:
cancer_df['label'] = y_encoded
cancer_df

In [ ]:
cancer_df = cancer_df.drop('id', axis=1)
cancer_df = cancer_df.drop('diagnosis', axis=1)
cancer_df

In [ ]:
# Amount values largely varies.

# # Normalization
# card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].min())/(card_df.iloc[:,:-1].max() - card_df.iloc[:,:-1].min())

# Standardization
cancer_df.iloc[:,:-1] = (cancer_df.iloc[:,:-1] - cancer_df.iloc[:,:-1].mean())/cancer_df.iloc[:,:-1].std()

cancer_df

In [ ]:
cancer_df['label'].value_counts()

In [ ]:
# Data is too balanced!!! We intentionally make it imbalanced.
df_0 = cancer_df[cancer_df['label']==0]
df_1 = cancer_df[cancer_df['label']==1]
print(len(df_0), len(df_1))

In [ ]:
N = round(len(df_0)*0.1)
df_1_samp = df_1.sample(n=N, random_state = 100)
df_1_samp

In [ ]:
cancer_df = concath(df_0, df_1_samp)
cancer_df

In [ ]:
cancer_df['label'].value_counts()

In [ ]:
T_sse= []
T_mse= []
T_bce= []
T_acc= []
T_f1= []
T_f05= []
T_f2= []
T_gmean= []
T_bacc= []
T_sola = []
T_solf = []
T_solb = []

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state = 2)

X = cancer_df.iloc[:, :-1]
y = cancer_df.iloc[:, -1]

In [ ]:
L = 73
hidden_node = 2
# momentum=0.9
activation = 'sigmoid'  
kernel_initializer=keras.initializers.he_normal(seed=100)
epochs=100

In [ ]:
n_iter=0

###################### MLP (sigmoid // MSE) ##############################
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate = 0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_mse = time.time()
    model.compile(loss=MSE, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_mse.append(time.time()-start_mse)
print(np.mean(T_mse))    

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_bce = time.time()
    model.compile(loss=BCE, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_bce.append(time.time()-start_bce)
print(np.mean(T_bce))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Accuracy) ##############################
batch_size = int(X.shape[0]*0.9 * 0.05)  # 0.05%0.004
print('batch_size: ', batch_size)
learning_rate = 0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_acc = time.time()
    model.compile(loss=Ours_Accu, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_acc.append(time.time()-start_acc)
print(np.mean(T_acc))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // F1) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   # 0.05&0.001
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 1
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f1 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f1.append(time.time()-start_f1)
print(np.mean(T_f1))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // F0.5) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.05&0.0005
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 0.5
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f05 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f05.append(time.time()-start_f05)
print(np.mean(T_f05))    

In [ ]:
n_iter=0
    
###################### MLP (sigmoid // sigmoid // F2) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.05&0.005
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 2
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)
    
for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f2 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f2.append(time.time()-start_f2)
print(np.mean(T_f2))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Gmean) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.5&0.005
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_gmean = time.time()
    model.compile(loss=Ours_Gmean, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_gmean.append(time.time()-start_gmean)
print(np.mean(T_gmean))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Balanced Accuracy) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.5&0.005  866
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_bacc = time.time()
    model.compile(loss=Ours_BAccu, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_bacc.append(time.time()-start_bacc)
print(np.mean(T_bacc))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_sola = time.time()
    model.compile(loss=SOL(score = 'accuracy', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_sola.append(time.time()-start_sola)
print(np.mean(T_sola))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_solf = time.time()
    model.compile(loss=SOL(score = 'f1_score', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_solf.append(time.time()-start_solf)
print(np.mean(T_solf))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(cancer_df, cancer_df['label']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_solb = time.time()
    model.compile(loss=SOL(score = 'bacc', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_solb.append(time.time()-start_solb)
print(np.mean(T_solb))  

In [ ]:
times = []
names = [T_mse, T_bce, T_acc, T_f1, T_f05, T_f2, T_gmean, T_bacc, T_sola, T_solf, T_solb]
NAME = ['T_mse', 'T_bce', 'T_acc', 'T_f1', 'T_f05', 'T_f2', 'T_gmean', 'T_bacc', 'T_sola', 'T_solf', 'T_solb']
for i in range(len(names)):
    times.append(np.mean(names[i]))

rate = times/times[1]

df_time = pd.DataFrame(NAME, columns=['names'])
df_time['times'] = times
df_time['rate'] = rate
df_time

In [ ]:
lmean = np.mean([times[2],times[3],times[7]])   # ours acc, f1, bacc
print(lmean, lmean/times[1])

In [ ]:
smean = np.mean(times[8:])   # sol acc, f1, bacc
print(smean, smean/times[1])

In [ ]:
np.mean(times[2:])

In [ ]:
np.mean(times[2:]) / times[1]

In [ ]:
import numpy as np
np.mean([3.615140, 3.501581, 3.669134, 3.635117, 3.655686, 3.489074])

In [ ]:
import numpy as np
np.mean([3.615140, 3.501581, 3.669134, 3.635117, 3.655686, 3.489074]) / 3.641144

## For Extension (2/5/25)


In [ ]:
## For Extension (2/5/25)
import numpy as np
# F1/GM/BA
# MSE/BCE/ANyLoss(Type1)
MSEtime = 3.158417
BCEtime = 3.847683
Anyf1 = 3.804214
Anygm = 3.427531
Anyba = 3.243834
print(MSEtime, BCEtime, Anyf1, Anygm, Anyba)
print(MSEtime/BCEtime, BCEtime/BCEtime, Anyf1/BCEtime, Anygm/BCEtime, Anyba/BCEtime)

# 4. Diabetes Prediction Data (8d / 100000)

In [ ]:
# class 'B' = Benign, class 'M' = Malignant
diab_df = pd.read_csv('diabetes_prediction_dataset.csv')
diab_df.shape

In [ ]:
diab_df.isnull().sum()

In [ ]:
diab_df

In [ ]:
# Female = 0, Male = 1, other = 2
gen_encoded, gen_class = pd.factorize(diab_df['gender'])
print(gen_class)
gen_encoded

In [ ]:
# Female = 0, Male = 1, other = 2
pd.Series(gen_encoded).value_counts()

In [ ]:
diab_df['gender'] = gen_encoded
diab_df

In [ ]:
# never = 0, Info = 1, current = 2, former=3, ever=4, not current=5
smo_encoded, smo_class = pd.factorize(diab_df['smoking_history'])
print(smo_class)
smo_encoded

In [ ]:
# never = 0, Info = 1, current = 2, former=3, ever=4, not current=5
pd.Series(smo_encoded).value_counts()

In [ ]:
diab_df['smoking_history'] = smo_encoded
diab_df

In [ ]:
diab_df.describe()

In [ ]:
# Amount values largely varies.

# # Normalization
# card_df.iloc[:,:-1] = (card_df.iloc[:,:-1] - card_df.iloc[:,:-1].min())/(card_df.iloc[:,:-1].max() - card_df.iloc[:,:-1].min())

# Standardization
diab_df.iloc[:,:-1] = (diab_df.iloc[:,:-1] - diab_df.iloc[:,:-1].mean())/diab_df.iloc[:,:-1].std()

diab_df

In [ ]:
T_sse= []
T_mse= []
T_bce= []
T_acc= []
T_f1= []
T_f05= []
T_f2= []
T_gmean= []
T_bacc= []
T_sola = []
T_solf = []
T_solb = []

In [ ]:
from sklearn.model_selection import StratifiedKFold
skf = StratifiedKFold(n_splits=10, shuffle=True, random_state = 2)

X = diab_df.iloc[:, :-1]
y = diab_df.iloc[:, -1]

In [ ]:
L = 73
hidden_node = 2
# momentum=0.9
activation = 'sigmoid'  
kernel_initializer=keras.initializers.he_normal(seed=100)
epochs=100

In [ ]:
n_iter=0

###################### MLP (sigmoid // MSE) ##############################
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate = 0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_mse = time.time()
    model.compile(loss=MSE, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_mse.append(time.time()-start_mse)
print(np.mean(T_mse))    

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_bce = time.time()
    model.compile(loss=BCE, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_bce.append(time.time()-start_bce)
print(np.mean(T_bce))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Accuracy) ##############################
batch_size = int(X.shape[0]*0.9 * 0.05)  # 0.05%0.004
print('batch_size: ', batch_size)
learning_rate = 0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_acc = time.time()
    model.compile(loss=Ours_Accu, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_acc.append(time.time()-start_acc)
print(np.mean(T_acc))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // F1) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   # 0.05&0.001
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 1
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f1 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f1.append(time.time()-start_f1)
print(np.mean(T_f1))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // F0.5) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.05&0.0005
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 0.5
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f05 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f05.append(time.time()-start_f05)
print(np.mean(T_f05))    

In [ ]:
n_iter=0
    
###################### MLP (sigmoid // sigmoid // F2) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.05&0.005
print('batch_size: ', batch_size)
learning_rate=0.005

beta = 2
model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)
    
for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_f2 = time.time()
    model.compile(loss=Ours_Fbeta, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_f2.append(time.time()-start_f2)
print(np.mean(T_f2))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Gmean) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.5&0.005
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_gmean = time.time()
    model.compile(loss=Ours_Gmean, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_gmean.append(time.time()-start_gmean)
print(np.mean(T_gmean))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // Balanced Accuracy) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)   #0.5&0.005  866
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_bacc = time.time()
    model.compile(loss=Ours_BAccu, optimizer=opt, metrics=['accuracy'])
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_bacc.append(time.time()-start_bacc)
print(np.mean(T_bacc))

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_sola = time.time()
    model.compile(loss=SOL(score = 'accuracy', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_sola.append(time.time()-start_sola)
print(np.mean(T_sola))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_solf = time.time()
    model.compile(loss=SOL(score = 'f1_score', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_solf.append(time.time()-start_solf)
print(np.mean(T_solf))  

In [ ]:
n_iter=0

###################### MLP (sigmoid // sigmoid // BCE) ############################## 
batch_size = int(X.shape[0]*0.9 * 0.05)  
print('batch_size: ', batch_size)
learning_rate=0.005

model = Sequential()
model.add(Dense(1, input_dim=X.shape[1], kernel_initializer=kernel_initializer))
opt = optimizers.Adam(learning_rate = learning_rate)   # SGD(learning_rate=learning_rate, momentum=momentum)

for train_index, test_index in skf.split(diab_df, diab_df['diabetes']):
    n_iter += 1
    X_train = X.iloc[train_index]
    y_train= y.iloc[train_index]
    X_test = X.iloc[test_index]
    y_test= y.iloc[test_index]
    print('#'*50,'{0}th CV'.format(n_iter),'#'*50)
    X_train = np.array(X_train)
    y_train = np.array(y_train)
    y_train = y_train.astype(float)
    X_test = np.array(X_test)
    y_test = np.array(y_test)
    y_test = y_test.astype(float)
    
    start_solb = time.time()
    model.compile(loss=SOL(score = 'bacc', distribution = 'cosine', mu = 0.5, delta = 0.1, mode = 'average'))
    history = model.fit(X_train, y_train, validation_data=(X_test, y_test), epochs=epochs, verbose=0, batch_size=batch_size)
    T_solb.append(time.time()-start_solb)
print(np.mean(T_solb))  

In [ ]:
times = []
names = [T_mse, T_bce, T_acc, T_f1, T_f05, T_f2, T_gmean, T_bacc, T_sola, T_solf, T_solb]
NAME = ['T_mse', 'T_bce', 'T_acc', 'T_f1', 'T_f05', 'T_f2', 'T_gmean', 'T_bacc', 'T_sola', 'T_solf', 'T_solb']
for i in range(len(names)):
    times.append(np.mean(names[i]))

rate = times/times[1]

df_time = pd.DataFrame(NAME, columns=['names'])
df_time['times'] = times
df_time['rate'] = rate
df_time

In [ ]:
lmean = np.mean([times[2],times[3],times[7]])   # ours acc, f1, bacc
print(lmean, lmean/times[1])

In [ ]:
smean = np.mean(times[8:])   # sol acc, f1, bacc
print(smean, smean/times[1])

In [ ]:
np.mean(times[2:])

In [ ]:
np.mean(times[2:]) / times[1]

In [ ]:
import numpy as np
np.mean([4.460501, 4.688376, 4.585820, 4.505197, 4.545006, 4.616062])

In [ ]:
import numpy as np
np.mean([4.460501, 4.688376, 4.585820, 4.505197, 4.545006, 4.616062]) / 4.538118

## For Extension (2/5/25)


In [ ]:
## For Extension (2/5/25)
import numpy as np
# F1/GM/BA
# MSE/BCE/ANyLoss(Type1)
MSEtime = 3.570375
BCEtime = 3.940880
Anyf1 = 4.327432
Anygm = 3.874397
Anyba = 4.062885
print(MSEtime, BCEtime, Anyf1, Anygm, Anyba)
print(MSEtime/BCEtime, BCEtime/BCEtime, Anyf1/BCEtime, Anygm/BCEtime, Anyba/BCEtime)